In [3]:
import json
import os
import random

In [4]:
# CONFIGURATION STRATIFY
DATA_ROOT = "./d2s_annotations_v1.1"
INPUT_JSON = os.path.join(DATA_ROOT, "annotations", "D2S_training.json")
OUTPUT_DIR = "./annotations"

def split_dataset_stratified():
    # Load the Data
    with open(INPUT_JSON, 'r') as f:
        data = json.load(f)

    images = data['images']
    annotations = data['annotations']
    categories = data['categories']
    print(f"Total Images found: {len(images)}")

    # Group Images by Scene (The first 4 digits of the ID)
    images.sort(key=lambda x: x['id'])

    # Perform the Split
    train_imgs = []
    val_imgs = []
    test_imgs = []

    batch_size = 10
    # Process the list in jumps of 10
    for i in range(0, len(images), batch_size):
        # Get batch of 10 images (or fewer if at the end)
        batch = images[i : i + batch_size]
        if len(batch) < batch_size:
            train_imgs.append(batch)
            continue

        # Create list of indices and shuffle it
        indices = list(range(len(batch)))
        random.shuffle(indices)

        # Assign based on shuffle position
        # 80% to training
        for idx in indices[:8]:
            train_imgs.append(batch[idx])
        # 10% to validation
        for idx in indices[8:9]:
            val_imgs.append(batch[idx])
        # 10% to testing
        for idx in indices[9:]:
            test_imgs.append(batch[idx])

    print(f"Split Complete.")
    print(f"Training Images: {len(train_imgs)} (Target: ~80%)")
    print(f"Validation Images: {len(val_imgs)} (Target: ~10%)")
    print(f"Test Images: {len(test_imgs)} (Target: ~10%)")

    # Filter Annotations
    train_img_ids = set(img['id'] for img in train_imgs)
    val_img_ids = set(img['id'] for img in val_imgs)
    test_img_ids = set(img['id'] for img in test_imgs)

    train_anns = [ann for ann in annotations if ann['image_id'] in train_img_ids]
    val_anns = [ann for ann in annotations if ann['image_id'] in val_img_ids]
    test_anns = [ann for ann in annotations if ann['image_id'] in test_img_ids]

    # Save Files
    def save_json(name, img_list, ann_list):
        out_data = {
            "info": data.get("info", {}),
            "licenses": data.get("licenses", []),
            "images": img_list,
            "annotations": ann_list,
            "categories": categories
        }
        with open(os.path.join(OUTPUT_DIR, name), 'w') as f:
            json.dump(out_data, f)
        print(f"Saved {name}")

    save_json("D2S_train_80.json", train_imgs, train_anns)
    save_json("D2S_val_10.json", val_imgs, val_anns)
    save_json("D2S_test_10.json", test_imgs, test_anns)
    print(f"Saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    split_dataset_stratified()

Total Images found: 4380
Split Complete.
Training Images: 3504 (Target: ~80%)
Validation Images: 438 (Target: ~10%)
Test Images: 438 (Target: ~10%)
Saved D2S_train_80.json
Saved D2S_val_10.json
Saved D2S_test_10.json
Saved to ./annotations
